# Polynomial Approximation: From Naive Least Squares to Modified Gram-Schmidt

## The Goal 

Find a polynomial $p(x) = c_0 + c_1 x + c_2 x^2 + \cdots + c_{n-1} x^{n-1}$ that approximates the function

$$f(x) = \frac{1}{1 + 25 x^2}$$

on the interval $[-1, 1]$ as well as possible.

This notebook walks through two ways to solve that problem:

1. **Naive least squares.** Sample $f$ at points $x_1, \ldots, x_N$, write down the polynomial-fitting linear system, and solve it. This is what every linear algebra textbook teaches first. We'll see it works fine at low degree but stalls badly at higher degree, refusing to improve no matter how many terms we add.
2. **Orthogonal projection via modified Gram-Schmidt.** We'll diagnose why the naive approach stalls (the columns of the design matrix become nearly linearly dependent), and use **modified Gram-Schmidt** to replace the monomial basis with an orthonormal one. The improvement is dramatic.

The story is essentially: *naive LSQ fails because monomials are a terrible basis at high degree, and MGS gives us a good one.*


## The Target Function

We'll work with $f(x) = 1 / (1 + 25 x^2)$ on $[-1, 1]$. This is the **Runge function**, a classical test case in numerical analysis. It's smooth and well-behaved on the real line, but it has poles at $x = \pm i / 5$ in the complex plane that make polynomial approximation harder than it looks.


In [ ]:
# Cell 01
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray


def f(x: NDArray) -> NDArray:
    return 1.0 / (1.0 + 25.0 * x**2)


N = 200
x = np.linspace(-1, 1, N)
y = f(x)

fig, ax = plt.subplots(figsize=(7, 4), num="Plot 1: Target Function")
ax.plot(x, y, color="black", linewidth=2.5, label="$f(x) = 1 / (1 + 25 x^2)$")
ax.axhline(0, color="black", linewidth=0.5, linestyle="dashed")
ax.set_title("Target function: the Runge function on $[-1, 1]$")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend()
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


## Approach 1: Naive Least Squares

The standard textbook setup. Write the polynomial in monomial form

$$p(x) = c_0 + c_1 x + c_2 x^2 + \cdots + c_{n-1} x^{n-1}$$

and arrange the data into a matrix equation. Define the **design matrix** $V$ where row $i$ holds the values of all $n$ monomials evaluated at sample point $x_i$:

$$V = \begin{pmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^{n-1} \\ 1 & x_2 & x_2^2 & \cdots & x_2^{n-1} \\ \vdots & & & & \vdots \\ 1 & x_N & x_N^2 & \cdots & x_N^{n-1} \end{pmatrix}$$

Then $V \mathbf{c} = \mathbf{y}$ would be exactly satisfied if $p$ went through every data point. But $V$ has $N = 200$ rows and only $n$ columns, so the system is overdetermined. Least squares finds the $\mathbf{c}$ that minimizes the squared residual $\|V \mathbf{c} - \mathbf{y}\|^2$, and the solution comes from the **normal equations**:

$$V^T V \, \mathbf{c} = V^T \mathbf{y}.$$

This is the "naive" approach because it uses the most direct formulation possible. It's also the one most students implement first.


In [ ]:
# Cell 02
def naive_lsq(x: NDArray, y: NDArray, n: int) -> tuple[NDArray, NDArray]:
    """Fit a degree (n-1) polynomial to (x, y) data via the normal equations.

    Parameters
    ----------
    x, y : NDArray
        Sample points and corresponding f(x) values.
    n : int
        Number of basis functions (1, x, ..., x^{n-1}).

    Returns
    -------
    V : NDArray, shape (len(x), n)
        Design matrix V[i, k] = x[i] ** k.
    c : NDArray, shape (n,)
        Coefficients in the monomial basis: p(x) = sum_k c[k] * x^k.
    """
    V = np.column_stack([x**k for k in range(n)])
    c = np.linalg.solve(V.T @ V, V.T @ y)
    return V, c


# Try it at low degree (n = 7 basis functions)
n_low = 7
V_low, c_low = naive_lsq(x, y, n_low)
p_low = V_low @ c_low

rmse_low = float(np.sqrt(np.mean((y - p_low) ** 2)))
print(f"Naive LSQ with n = {n_low}: RMSE = {rmse_low:.4e}")

fig, ax = plt.subplots(figsize=(7, 4), num="Plot 2: Naive LSQ at n=7")
ax.plot(x, y, color="black", linewidth=2.5, label="$f(x)$")
ax.plot(x, p_low, color="orangered", linewidth=1.8, label=f"naive LSQ, $n = {n_low}$")
ax.axhline(0, color="black", linewidth=0.5, linestyle="dashed")
ax.set_title(
    f"Naive least squares with {n_low} basis functions  (RMSE = {rmse_low:.3e})"
)
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend()
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


### What if we use more terms?

Naive intuition: more basis functions, better fit. Doubling the basis from $n = 7$ to $n = 15$ should let the polynomial follow the peak more tightly. We'll try a range of higher $n$ and tabulate both the fit quality (RMSE) and a quantity called the **condition number** of $V^T V$.

#### What the condition number is, why it matters

For a square matrix $A$, the **condition number** $\kappa$ measures how much the solution of $A \mathbf{c} = \mathbf{b}$ can change in response to a small perturbation in $\mathbf{b}$. The standard definition is

$$\kappa(A) \;=\; \|A\|\, \|A^{-1}\| \;=\; \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)},$$

the ratio of the largest to the smallest singular value of $A$. (This is what `np.linalg.cond(A)` returns by default. It computes the SVD and takes that ratio.) A perfectly conditioned matrix has $\kappa = 1$; the identity is the canonical example. As $A$ becomes nearly singular (meaning its columns become close to linearly dependent), the smallest singular value $\sigma_{\min}$ shrinks toward zero and $\kappa$ blows up.

Why this matters in practice: a relative perturbation of size $\varepsilon$ in $\mathbf{b}$ produces a relative error in the solution $\mathbf{c}$ bounded by $\kappa(A) \cdot \varepsilon$. In double-precision floating point every number is stored with a relative round-off error around $\varepsilon_{\text{mach}} \approx 2.2 \times 10^{-16}$. So once $\kappa(A) \cdot \varepsilon_{\text{mach}} \gtrsim 1$, which happens at $\kappa(A) \gtrsim 10^{16}$, the relative error in the computed solution reaches order one. At that point the answer is <span style="color: red">**meaningless**</span>. The floating-point representation itself has used up all the precision the data carried, before the algorithm even gets a chance.

For naive LSQ we are solving $V^T V \, \mathbf{c} = V^T \mathbf{y}$, so $\kappa(V^T V)$ is the number to watch as $n$ grows.


In [ ]:
# Cell 03
ns_naive = [7, 11, 15, 19, 23, 27, 31]
naive_results = []

print(f"{'n':>3}  {'RMSE':>11}  {'cond(V^T V)':>13}")
print("-" * 32)
for n in ns_naive:
    V_n, c_n = naive_lsq(x, y, n)
    p_n = V_n @ c_n
    rmse = float(np.sqrt(np.mean((y - p_n) ** 2)))
    cond = float(np.linalg.cond(V_n.T @ V_n))
    naive_results.append((n, p_n, rmse, cond))
    print(f"{n:>3}  {rmse:>11.3e}  {cond:>13.2e}")


### Plot 3: Naive LSQ stops improving

Three fits at $n = 7$, $19$, and $31$ overlaid on $f$. The $n = 7$ fit (light) is the rough one we saw before. Adding more terms helps initially. By $n = 19$ the fit tracks the peak much better. By $n = 31$, however, the improvement over $n = 19$ is barely visible. The RMSE table tells the same story: the error keeps decreasing but the rate slows dramatically, and after $n \approx 23$ the improvement essentially stops. Adding more basis functions is no longer helping.

Why?


In [ ]:
# Cell 04
fig, ax = plt.subplots(figsize=(7, 4.2), num="Plot 3: Naive LSQ stalls")

ax.plot(x, y, color="black", linewidth=2.5, label="$f(x)$")
shown = [(7, "royalblue"), (19, "forestgreen"), (31, "crimson")]
for n_show, color in shown:
    p_show = next(p for (nn, p, _, _) in naive_results if nn == n_show)
    rmse_show = next(r for (nn, _, r, _) in naive_results if nn == n_show)
    ax.plot(
        x,
        p_show,
        color=color,
        linewidth=1.6,
        label=f"$n = {n_show}$  (RMSE = {rmse_show:.2e})",
    )

ax.axhline(0, color="black", linewidth=0.5, linestyle="dashed")
ax.set_title("Adding more naive-LSQ basis functions stops helping")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend(fontsize=9)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


## Why It Stalls: The Matrix $V^T V$ Becomes Singular

Look at the `cond(V^T V)` column from the table. It starts at $\sim\!10^4$ for $n = 7$ and grows by about a factor of $10^3$ for every 4 more basis functions. By $n = 23$ it's around $10^{16}$. That's the same order as machine epsilon for double-precision floating point ($2.2 \times 10^{-16}$).

Once $\kappa(V^T V) \cdot \epsilon_\text{mach} \gtrsim 1$, the linear solver $\mathbf{c} = (V^T V)^{-1} V^T \mathbf{y}$ is computing nonsense. Round-off errors of order machine epsilon get amplified by the condition number, swamping the actual signal. The polynomial $V \mathbf{c}$ that comes back is no longer the true least-squares solution, and additional basis functions can't help because the linear system is what's broken.

The diagnosis points to the cause: $V^T V$ is **the matrix of pairwise dot products of the monomial column vectors**. Its $(i, j)$ entry is

$$\bigl(V^T V\bigr)_{ij} = \sum_{k=1}^{N} x_k^{i}\, x_k^{j}.$$

When this matrix is nearly singular, it means the columns of $V$ (the monomials sampled at our points) are nearly linearly dependent. Their pairwise dot products are large, the columns "point in similar directions" in $\mathbb{R}^N$, and the system $V^T V \mathbf{c} = V^T \mathbf{y}$ becomes hard to solve. **The monomials are a bad basis, not because they're wrong, but because at high degree they're nearly redundant.**

The cure is suggested by the diagnosis: replace the monomials with a basis whose columns are *not* nearly linearly dependent. Specifically, we want a basis whose column vectors are **orthogonal**. That means columns whose pairwise dot products are zero. In that case $V^T V$ would be (a multiple of) the identity matrix. Perfectly conditioned. No solver issues.


## Modified Gram-Schmidt: Building an Orthogonal Basis

Two column vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^N$ are **orthogonal** when their dot product is zero: $\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i = 0$. **Modified Gram-Schmidt** takes a set of linearly independent vectors and constructs an orthonormal set spanning the same subspace, using a clean recurrence:

For $k = 0, 1, 2, \ldots$:
1. Subtract from $\mathbf{v}_k$ all of its components along the previously computed $\mathbf{e}_0, \ldots, \mathbf{e}_{k-1}$.
2. Normalize what's left: $\mathbf{e}_k = \mathbf{v}_k / \|\mathbf{v}_k\|$.

The "modified" part is that step 1 is done **incrementally**. We update $\mathbf{v}_k$ in place after each subtraction, rather than computing all subtractions from the original $\mathbf{v}_k$. This is mathematically equivalent in exact arithmetic but more stable numerically.

We apply MGS to the columns of our monomial design matrix $V$, producing a new matrix $Q$ whose columns are orthonormal. Crucially, $Q$ spans the same column space as $V$. Every polynomial expressible in monomials is also expressible in the new basis.


In [ ]:
# Cell 05
def modified_gram_schmidt(V: NDArray) -> NDArray:
    """Orthonormalize the columns of V using modified Gram-Schmidt.

    Parameters
    ----------
    V : NDArray, shape (m, n)
        Matrix whose columns are the input vectors.

    Returns
    -------
    Q : NDArray, shape (m, n)
        Matrix with orthonormal columns: Q.T @ Q = I_n. The column space
        of Q equals the column space of V.
    """
    Q = V.astype(np.float64).copy()
    _, n = Q.shape
    for i in range(n):
        # Normalize column i to unit length
        Q[:, i] /= np.linalg.norm(Q[:, i])
        # Subtract its component from all subsequent columns (in place)
        for j in range(i + 1, n):
            Q[:, j] -= np.dot(Q[:, j], Q[:, i]) * Q[:, i]
    return Q


# Apply to the same monomial design matrix at n = 7 and verify orthonormality
n_basis = 7
V_basis = np.column_stack([x**k for k in range(n_basis)])
Q_basis = modified_gram_schmidt(V_basis)

QtQ = Q_basis.T @ Q_basis
err = float(np.max(np.abs(QtQ - np.eye(n_basis))))
print(f"Q.T @ Q - I  : max abs entry = {err:.2e}  (machine epsilon ~ 2.2e-16)")
print(f"Column norms : {np.round(np.linalg.norm(Q_basis, axis=0), 12)}")


### Plot 4: The orthogonal basis

The columns of $Q$, plotted as functions of $x$. Each one is a polynomial, specifically a linear combination of $1, x, x^2, \ldots, x^6$. They are mutually orthogonal under the dot product. The patterns ($e_k$ has $k$ zero crossings, the parity of $e_k$ matches the parity of $k$) come straight from the orthogonality requirement.

These curves happen to be (essentially) the **Legendre polynomials**, a well-known orthogonal family on $[-1, 1]$ that shows up all over numerical analysis. We didn't ask for Legendre polynomials, just for an orthogonal basis of the polynomial space; they emerged automatically because the orthogonality requirement determines them up to sign.


In [ ]:
# Cell 06
basis_colors = [
    "crimson",
    "darkorange",
    "goldenrod",
    "forestgreen",
    "steelblue",
    "darkslateblue",
    "purple",
]

fig, ax = plt.subplots(figsize=(7, 4.2), num="Plot 4: Orthogonal Basis")
for k in range(n_basis):
    ax.plot(x, Q_basis[:, k], color=basis_colors[k], linewidth=2, label=f"$e_{{{k}}}$")
ax.axhline(0, color="black", linewidth=0.5, linestyle="dashed")
ax.set_title(
    f"Orthonormal basis from MGS applied to $\\{{1, x, ..., x^{{{n_basis - 1}}}\\}}$"
)
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend(loc="upper left", ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


## Approach 2: Orthogonal Projection

With an orthonormal basis $Q$ in hand, the least-squares problem becomes trivial. The normal equations are still $Q^T Q \, \mathbf{c} = Q^T \mathbf{y}$, but now $Q^T Q = I$, so

$$\mathbf{c} = Q^T \mathbf{y}.$$

Each coefficient is one independent dot product:

$$c_k = \mathbf{e}_k \cdot \mathbf{y} = \sum_i e_k(x_i)\, y_i.$$

There is no linear system to solve and no $V^T V$ to invert. The conditioning issue that broke naive LSQ is gone. We replaced "solve a linear system" with "compute $n$ separate dot products," and dot products don't have conditioning problems.


In [ ]:
# Cell 07
def ortho_fit(x: NDArray, y: NDArray, n: int) -> tuple[NDArray, NDArray]:
    """Fit a degree (n-1) polynomial to (x, y) by orthogonal projection.

    Parameters
    ----------
    x, y : NDArray
        Sample points and f(x) values.
    n : int
        Number of basis functions.

    Returns
    -------
    Q : NDArray, shape (len(x), n)
        Orthonormal basis (MGS applied to monomial design matrix).
    c : NDArray, shape (n,)
        Coefficients in the orthonormal basis: p(x_i) = sum_k c[k] * Q[i, k].
    """
    V = np.column_stack([x**k for k in range(n)])
    Q = modified_gram_schmidt(V)
    c = Q.T @ y
    return Q, c


# Apply at the same n values we tried with naive LSQ
ns_ortho = ns_naive  # [7, 11, 15, 19, 23, 27, 31]
ortho_results = []

print(f"{'n':>3}  {'RMSE':>11}")
print("-" * 16)
for n in ns_ortho:
    Q_n, c_n = ortho_fit(x, y, n)
    p_n = Q_n @ c_n
    rmse = float(np.sqrt(np.mean((y - p_n) ** 2)))
    ortho_results.append((n, p_n, rmse))
    print(f"{n:>3}  {rmse:>11.3e}")


### Plot 5: Orthogonal projection keeps improving

Same three $n$ values as Plot 3. Unlike the naive method, every increase in $n$ tightens the fit visibly, and there's no stalling: the RMSE drops by a factor of ~6 from $n = 7$ to $n = 31$. The fit at $n = 31$ is essentially indistinguishable from $f$ at this resolution.


In [ ]:
# Cell 08
fig, ax = plt.subplots(figsize=(7, 4.2), num="Plot 5: Orthogonal projection improves")

ax.plot(x, y, color="black", linewidth=2.5, label="$f(x)$")
shown = [(7, "royalblue"), (19, "forestgreen"), (31, "crimson")]
for n_show, color in shown:
    p_show = next(p for (nn, p, _) in ortho_results if nn == n_show)
    rmse_show = next(r for (nn, _, r) in ortho_results if nn == n_show)
    ax.plot(
        x,
        p_show,
        color=color,
        linewidth=1.6,
        label=f"$n = {n_show}$  (RMSE = {rmse_show:.2e})",
    )

ax.axhline(0, color="black", linewidth=0.5, linestyle="dashed")
ax.set_title("Orthogonal projection on MGS basis: every term helps")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend(fontsize=9)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


## Numerical Comparison

Run both methods over a wider range of $n$ and tabulate the resulting RMSE side by side. This is the apples-to-apples comparison: identical sample points, identical target function, identical underlying basis space at each $n$. The only difference is the algorithm used to compute the coefficients.


In [ ]:
# Cell 09
ns_compare = list(range(3, 36, 2))

print(
    f"{'n':>3}  {'naive RMSE':>12}  {'ortho RMSE':>12}  {'naive/ortho':>12}  {'cond(V^TV)':>11}"
)
print("-" * 60)

err_naive_arr = []
err_ortho_arr = []
cond_arr = []

for n in ns_compare:
    V_n, c_n_naive = naive_lsq(x, y, n)
    p_naive = V_n @ c_n_naive
    err_naive = float(np.sqrt(np.mean((y - p_naive) ** 2)))
    cond = float(np.linalg.cond(V_n.T @ V_n))

    Q_n, c_n_ortho = ortho_fit(x, y, n)
    p_ortho = Q_n @ c_n_ortho
    err_ortho = float(np.sqrt(np.mean((y - p_ortho) ** 2)))

    err_naive_arr.append(err_naive)
    err_ortho_arr.append(err_ortho)
    cond_arr.append(cond)

    ratio = err_naive / err_ortho if err_ortho > 0 else float("inf")
    print(
        f"{n:>3}  {err_naive:>12.3e}  {err_ortho:>12.3e}  {ratio:>12.1f}  {cond:>11.2e}"
    )


### Plot 6: Convergence comparison

The two RMSE curves on a log scale, against $n$. Through about $n = 23$ they overlap exactly. Both methods are computing the same answer because the conditioning hasn't broken anything yet. Past that point the naive method flattens out at RMSE $\approx 5 \times 10^{-3}$, refusing to improve no matter how many basis functions are added. The orthogonal method continues its straight-line descent until even MGS itself runs out of precision around $n \approx 35$.

The horizontal naive-LSQ plateau is exactly the symptom we saw in Plot 3, now quantified. The vertical separation between the two curves at any given $n > 25$ is the practical price of using a non-orthogonal basis at high degree.

This is the operational reason orthogonal polynomial families (Legendre, Chebyshev, Hermite, Laguerre) appear everywhere in numerical analysis: they sidestep the conditioning problem by construction. MGS is the constructive bridge. Given any reasonable starting basis, it produces an orthogonal one that you can use without paying the conditioning tax.


In [ ]:
# Cell 10
fig, ax = plt.subplots(figsize=(7.5, 4.5), num="Plot 6: Convergence")

ax.semilogy(
    ns_compare,
    err_naive_arr,
    color="orangered",
    marker="o",
    linewidth=2,
    label="Naive LSQ (normal equations on monomials)",
)
ax.semilogy(
    ns_compare,
    err_ortho_arr,
    color="mediumseagreen",
    marker="s",
    linewidth=2,
    label="Orthogonal projection (MGS basis)",
)

# Mark where condition number crosses machine epsilon
eps_machine = 2.2e-16
crossover_idx = next((i for i, c in enumerate(cond_arr) if c * eps_machine > 1), None)
if crossover_idx is not None:
    n_cross = ns_compare[crossover_idx]
    ax.axvline(n_cross, color="dimgray", linestyle="dashed", linewidth=1, alpha=0.7)
    ax.text(
        n_cross + 0.3,
        0.25,
        f"$\\kappa(V^T V) \\cdot \\epsilon_\\mathrm{{mach}} > 1$\nat $n = {n_cross}$",
        fontsize=8,
        color="dimgray",
        verticalalignment="top",
    )

ax.set_title(
    "Approximation error vs basis size for $f(x) = 1/(1 + 25 x^2)$\n"
    "(both methods identical until conditioning kills the naive solver)"
)
ax.set_xlabel("number of basis functions $n$")
ax.set_ylabel("RMSE  $\\sqrt{\\mathrm{mean}((y - p)^2)}$")
ax.legend(loc="lower left", fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()
